# 01. システムズシンキング（Systems Thinking） — 練習問題

**対象技術**: 量子コンピューティング

技術の波及効果を「符号付き因果ループ図（causal loop diagram）」として表現し、閉ループ（フィードバックループ）を列挙して、各ループを構成するリンク符号の積から強化ループ（reinforcing, R）・均衡ループ（balancing, B）を判定する手法である。ループ構造を機械的に解析することで、少ない介入で構造全体を動かせるレバレッジ変数を見つける手がかりが得られる。

必要なライブラリを読み込む。`numpy` で符号付き隣接行列を扱い、`matplotlib` で因果ループ図を描画する。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


## モデル定義

量子コンピューティング普及の因果ループ図を構成する変数を列挙する。抽象概念をそのまま変数にせず、反復して観測できる代理指標（proxy）と単位を割り当てる。たとえば「暗号危殆化への懸念」は直接測れないため、定期調査で『量子暗号リスクを経営課題として認知している』と答えた企業の割合で測る。「蓄積実行データ量」はストック変数である。

| ID | 定量指標 | 単位 | 測定方法 |
|---|---|---|---|
| V0 | 量子クラウド利用量 | QPU時間/月 | クラウド事業者の月次実行ログ |
| V1 | 蓄積実行データ量 | GB | 再利用可能な実行結果・校正・ベンチマークデータの累積容量 |
| V2 | アルゴリズム性能 | % | 固定した基準問題群の成功率 |
| V3 | 量子人材需要 | 件/月 | 量子スキルを要件に含む月次求人数 |
| V4 | 量子人材賃金 | 万円/年 | 対象職種の年間報酬中央値 |
| V5 | 量子クラウド単価 | 円/QPU時間 | 同一性能帯の実効利用料金 |
| V6 | FTQC実現度 | 個 | 誤り訂正済み論理量子ビット数 |
| V7 | 暗号リスク認知 | % | リスクを経営課題と認知する企業の回答率 |
| V8 | PQC移行投資 | 億円/年 | 調査対象組織の年間移行支出合計 |
| V9 | 暗号危殆化の社会的影響 | 億円/年 | シナリオ別の推定年間損失額 |

測定周期は原則として月次または年次に統一する。本Notebookの年次ショックモデルでは実測値そのものを加算せず、各指標について「ショックなしケースとの差」をポイント表示し、リンクの伝播係数によって次の変数へ渡す。したがって出力は実数値の予測ではなく、共通シナリオに対する相対応答である。

In [ ]:
VARIABLES = [
    "量子クラウド利用量",          # 0
    "蓄積実行データ量",            # 1 (ストック変数)
    "基準問題成功率",              # 2
    "量子人材求人数",              # 3
    "量子人材年間報酬中央値",      # 4
    "量子クラウド実効単価",        # 5
    "誤り訂正済み論理量子ビット数",# 6
    "暗号リスク認知企業率",        # 7
    "PQC年間移行投資額",           # 8
    "暗号危殆化推定年間損失額",    # 9
]

UNITS = [
    "QPU時間/月", "GB", "%", "件/月", "万円/年",
    "円/QPU時間", "個", "%", "億円/年", "億円/年",
]

MEASUREMENT = [
    "クラウド事業者の月次実行ログ",
    "再利用可能な実行結果・校正・ベンチマークデータの累積容量",
    "固定した基準問題群の成功率",
    "量子スキルを要件に含む月次求人数",
    "対象職種の年間報酬中央値",
    "同一性能帯の実効利用料金",
    "誤り訂正を維持して計算に使える論理量子ビット数",
    "リスクを経営課題と認知する企業の定期調査回答率",
    "調査対象組織の年間PQC移行支出合計",
    "資産曝露額×侵害確率×損失率によるシナリオ別推定値",
]

for i, (v, unit, measure) in enumerate(zip(VARIABLES, UNITS, MEASUREMENT)):
    print(f"  V{i}: {v} [{unit}]")
    print(f"      測定: {measure}")

全分析で共通利用するリンクを `(原因, 結果): {sign, effect, delay}` で定義する。`sign` は因果の極性、`effect` は新しく到着した変化の伝播係数、`delay` は結果へ届くまでの年数である。閉路検出と因果ループ図に必要な `(原因, 結果, 極性)` は、この共通定義から自動生成する。

In [ ]:
# 共通リンク定義: 構造・年次伝播・感度分析の全てがこの辞書を参照する。
# sign: 因果の極性 / effect: 新規変化の伝播係数 / delay: 到着までの年数
LINKS = {
    # 強化ループ R1: 利用 -> データ蓄積 -> 成功率 -> 利用
    (0, 1): {"sign": +1, "effect": 0.70, "delay": 1},
    (1, 2): {"sign": +1, "effect": 0.25, "delay": 2},
    (2, 0): {"sign": +1, "effect": 0.40, "delay": 1},
    # 均衡ループ B1: 利用 -> 求人数 -> 報酬 -> 単価 -> 利用
    (0, 3): {"sign": +1, "effect": 0.35, "delay": 1},
    (3, 4): {"sign": +1, "effect": 0.15, "delay": 1},
    (4, 5): {"sign": +1, "effect": 0.20, "delay": 2},
    (5, 0): {"sign": -1, "effect": 0.30, "delay": 1},
    # 均衡ループ B2: リスク認知 -> PQC投資 -> 損失低下 -> リスク認知
    (6, 7): {"sign": +1, "effect": 0.20, "delay": 1},
    (7, 8): {"sign": +1, "effect": 0.50, "delay": 1},
    (8, 9): {"sign": -1, "effect": 0.60, "delay": 2},
    (9, 7): {"sign": +1, "effect": 0.25, "delay": 1},
}

# 閉路検出・因果ループ図には符号だけを取り出して使う。
EDGES = [(src, dst, p["sign"]) for (src, dst), p in LINKS.items()]
print(f"変数 {len(VARIABLES)} 個 / リンク {len(LINKS)} 本")

## ショックの年次伝播

ここでは全変数が毎年一定率で成長するとは仮定しない。通常ケースでは全変数の変化をゼロとし、特定の年に1つの変数へ加わった変化（ショック）が、リンクごとの遅延を伴って隣接変数へ順番に届く過程を計算する。表示する値は通常ケースとの差、すなわち**ショックによって追加的に生じた変化（%ポイント）**である。

`effect=0.40` は、原因変数へ届いた `+10` ポイントの新しい変化が、結果変数へ `+4` ポイント伝わるという意味である。負リンクなら符号が反転する。`delay=2` なら、その効果が結果変数へ現れるのは2年後である。各年に新しく到着した変化だけを次の変数へ送り、すでに到着した影響は `PERSISTENCE` の割合だけ翌年へ残る。これにより、一度の変化が全変数を同時に右肩上がりにするのではなく、玉突き事故やドミノ倒しのように時間差で波及する。

伝播係数・遅延・残存率は初期シナリオ仮定であり、実証分析では時系列データや専門家調査で校正する。

In [ ]:
START_YEAR = 2025
END_YEAR = 2040
YEARS = np.arange(START_YEAR, END_YEAR + 1)

# 因果的に独立した2系統を、それぞれの起点へのショックで観察する。
SHOCK_SCENARIOS = [
    {"name": "利用拡大ショック", "year": 2027, "actor": 0, "size": 20.0},
    {"name": "FTQC進展ショック", "year": 2027, "actor": 6, "size": 20.0},
]
PERSISTENCE = 0.45  # 到着済みの影響が翌年に残る割合

print("ショック・シナリオ")
for s in SHOCK_SCENARIOS:
    print(f"  {s['year']}年 V{s['actor']} {VARIABLES[s['actor']]} "
          f"{s['size']:+.1f}ポイント")
print(f"影響の翌年残存率: {PERSISTENCE:.0%}")

print("\n共通リンク・パラメータ")
for (src, dst), p in LINKS.items():
    print(f"  V{src}->V{dst} ({'+' if p['sign'] > 0 else '-'}) "
          f"伝播係数={p['effect']:.2f}, 遅延={p['delay']}年")

In [ ]:
def simulate_shock_propagation(years, n_vars, links,
                               shock_year, shock_actor, shock_size,
                               persistence=0.45):
    """共通LINKSを使い、単発ショックの年次伝播を計算する。

    pulse[t, i]       : t年に変数iへ新しく到着した変化
    impact[t, i]      : 過年度から残った影響を含む通常ケースとの差
    edge_pulse[t, e]  : t年にリンクeから新しく到着した寄与
    edge_impact[t, e] : 過年度から残ったリンク別寄与
    """
    n_years = len(years)
    link_items = list(links.items())
    pulse = np.zeros((n_years, n_vars))
    impact = np.zeros((n_years, n_vars))
    edge_pulse = np.zeros((n_years, len(link_items)))
    edge_impact = np.zeros((n_years, len(link_items)))

    matches = np.where(years == shock_year)[0]
    if len(matches) != 1:
        raise ValueError("shock_year は YEARS に1回だけ含まれる必要があります")
    pulse[int(matches[0]), shock_actor] = shock_size

    for t in range(n_years):
        if t == 0:
            impact[t] = pulse[t]
            edge_impact[t] = edge_pulse[t]
        else:
            impact[t] = persistence * impact[t - 1] + pulse[t]
            edge_impact[t] = persistence * edge_impact[t - 1] + edge_pulse[t]

        # この年に新しく到着した変化だけを、遅延付きで下流へ送る。
        for e, ((src, dst), p) in enumerate(link_items):
            arrival_t = t + p["delay"]
            if arrival_t < n_years:
                flow = p["sign"] * p["effect"] * pulse[t, src]
                pulse[arrival_t, dst] += flow
                edge_pulse[arrival_t, e] += flow

    return {
        "pulse": pulse,
        "impact": impact,
        "edge_pulse": edge_pulse,
        "edge_impact": edge_impact,
        "link_keys": [key for key, _ in link_items],
    }


scenario_results = []
for scenario in SHOCK_SCENARIOS:
    result = simulate_shock_propagation(
        YEARS, len(VARIABLES), LINKS,
        scenario["year"], scenario["actor"], scenario["size"], PERSISTENCE
    )
    scenario_results.append({**scenario, **result})

    print(f"\n[{scenario['name']}] 年ごとに新しく到着する波及（絶対値0.05以上）")
    for t, year in enumerate(YEARS):
        arrived = [f"V{i} {result['pulse'][t, i]:+.2f}"
                   for i in range(len(VARIABLES))
                   if abs(result["pulse"][t, i]) >= 0.05]
        if arrived:
            print(f"  {year}: " + ", ".join(arrived))

In [ ]:
fig, axes = plt.subplots(
    1, len(scenario_results), figsize=(19, 7), sharex=True, sharey=True
)
axes = np.atleast_1d(axes)

# 2シナリオを同じ色尺度で比較する。初期ショックが大きいため、
# 非ゼロ値の80パーセンタイルで表示範囲を切り、微小な波及も見えるようにする。
nonzero = np.concatenate([
    np.abs(r["pulse"][np.abs(r["pulse"]) >= 0.05])
    for r in scenario_results
])
color_limit = max(1.0, float(np.percentile(nonzero, 80)))

for ax, result in zip(axes, scenario_results):
    pulse = result["pulse"]
    im = ax.imshow(
        pulse.T, aspect="auto", cmap="RdBu_r",
        vmin=-color_limit, vmax=color_limit, interpolation="nearest"
    )
    ax.set_xticks(np.arange(len(YEARS)))
    ax.set_xticklabels(YEARS, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(VARIABLES)))
    ax.set_yticklabels([f"V{i}: {name}" for i, name in enumerate(VARIABLES)])
    ax.set_xlabel("year")
    ax.set_title(
        f"{result['name']}\n"
        f"V{result['actor']} in {result['year']}: {result['size']:+.0f} points"
    )

    for t in range(len(YEARS)):
        for i in range(len(VARIABLES)):
            if abs(pulse[t, i]) >= 0.05:
                ax.add_patch(plt.Rectangle(
                    (t - 0.5, i - 0.5), 1, 1,
                    fill=False, edgecolor="#222222", lw=1.4
                ))
                ax.text(
                    t, i, f"{pulse[t, i]:+.1f}",
                    ha="center", va="center", fontsize=7.5,
                    color="#111111", fontweight="bold"
                )

axes[0].set_ylabel("quantified variable")
fig.suptitle(
    "Shock propagation by causal subsystem\n"
    "red = increase, blue = decrease, white = no newly arriving impact",
    fontsize=13
)
cbar = fig.colorbar(im, ax=axes.tolist(), pad=0.02, shrink=0.88)
cbar.set_label("new impact arriving in that year (percentage points)")
plt.show()

print("[読み方]")
print("  左: V0からV1〜V5へ波及する利用・人材・価格系統。")
print("  右: V6からV7〜V9へ波及するFTQC・暗号リスク・PQC系統。")
print("  白い行は、そのショックから到達する因果経路がないことを示す。")
print("  2系統をつなぐ根拠がないため、別シナリオとして比較している。")
print("  色は見やすさのため上位値で飽和する。正確な到着量はセル内の数値で読む。")

## 解析関数

符号付き隣接行列の構築、深さ優先探索（DFS）による全単純閉路の列挙、符号積によるループ分類、負リンク本数の計数を行う関数を定義する。

In [ ]:
def build_adjacency(n, edges):
    """符号付き隣接行列を構築する。A[i, j] が i->j のリンク極性。"""
    A = np.zeros((n, n), dtype=int)
    for src, dst, sign in edges:
        A[src, dst] = sign
    return A


def find_all_cycles(A):
    """深さ優先探索で全ての単純閉路を列挙する。

    各閉路は最小ノード番号から始まる正規形にして重複を排除する。
    戻り値は (ノード列, 符号積) のリスト。
    """
    n = A.shape[0]
    cycles = []
    seen = set()

    def dfs(start, current, path, sign_product):
        for nxt in range(n):
            s = A[current, nxt]
            if s == 0:
                continue
            if nxt == start:
                full_sign = sign_product * s
                k = path.index(min(path))
                norm = tuple(path[k:] + path[:k])
                if norm not in seen:
                    seen.add(norm)
                    cycles.append((list(norm), full_sign))
            elif nxt > start and nxt not in path:
                dfs(start, nxt, path + [nxt], sign_product * s)

    for start in range(n):
        dfs(start, start, [start], 1)
    return cycles


def classify_cycle(sign_product):
    """符号積が正なら強化ループ(R)、負なら均衡ループ(B)。"""
    return "強化ループ R" if sign_product > 0 else "均衡ループ B"


def count_negative_links(A, nodes):
    """ループ内の負リンク本数を数える(分類の検算用)。"""
    neg = 0
    for i in range(len(nodes)):
        src = nodes[i]
        dst = nodes[(i + 1) % len(nodes)]
        if A[src, dst] < 0:
            neg += 1
    return neg

## 全閉路の列挙と分類

隣接行列を構築し、全ての閉ループを列挙して R / B に分類する。各ループの経路・符号積・負リンク数を表示する。

In [ ]:
n = len(VARIABLES)
A = build_adjacency(n, EDGES)
cycles = find_all_cycles(A)

print(f"検出された閉ループ: {len(cycles)} 本")
print("-" * 68)

loop_participation = np.zeros(n, dtype=int)  # 各変数のループ参加数
r_count = b_count = 0
for idx, (nodes, sign) in enumerate(cycles, start=1):
    kind = classify_cycle(sign)
    if sign > 0:
        r_count += 1
    else:
        b_count += 1
    neg = count_negative_links(A, nodes)
    names = " -> ".join(VARIABLES[v] for v in nodes)
    print(f"ループ{idx} [{kind}]")
    print(f"  経路 : {names} -> (先頭へ)")
    print(f"  符号積={sign:+d}  負リンク数={neg}({'偶数=R' if neg % 2 == 0 else '奇数=B'})")
    for v in nodes:
        loop_participation[v] += 1
print("-" * 68)
print(f"内訳: 強化ループ R = {r_count} 本 / 均衡ループ B = {b_count} 本")

## レバレッジ候補の素朴な指標

各変数が何本のループを通過するかを集計する。多くのループを通過する変数は、そこへの介入が複数ループに同時作用するためレバレッジが高い。

In [ ]:
print("各変数のループ参加数(多いほどレバレッジ候補):")
order = np.argsort(-loop_participation)
for v in order:
    bar = "#" * loop_participation[v]
    print(f"  {VARIABLES[v]:<22} {loop_participation[v]:>2} {bar}")

top = VARIABLES[order[0]]
print()
print("[解釈]")
print(f"  最も多くのループを通過する変数は『{top}』。")
print("  ここへの介入は複数ループに同時作用するためレバレッジが高い。")

## 可視化: 因果ループ図

変数ノードを円周上に配置し、リンクを矢印で描く。極性 + は実線・暖色、− は破線・寒色で色分けし、検出した強化/均衡ループの本数を注記する。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# 変数ノードを円周上に配置
angles = np.linspace(np.pi / 2, np.pi / 2 + 2 * np.pi, n, endpoint=False)
pos = np.column_stack([np.cos(angles), np.sin(angles)])

# リンク(矢印)を描画: + は実線/暖色, - は破線/寒色
for src, dst, sign in EDGES:
    x0, y0 = pos[src]
    x1, y1 = pos[dst]
    color = "#d1495b" if sign > 0 else "#3a6ea5"
    style = "-" if sign > 0 else "--"
    ax.annotate(
        "", xy=(x1 * 0.82, y1 * 0.82), xytext=(x0 * 0.82, y0 * 0.82),
        arrowprops=dict(arrowstyle="-|>", color=color, lw=1.8,
                        linestyle=style, shrinkA=14, shrinkB=14,
                        connectionstyle="arc3,rad=0.15"))

# ノードを描画
for i, (x, y) in enumerate(pos):
    ax.scatter(x, y, s=900, c="#f4e285", edgecolors="#5a5a5a", zorder=3)
    ax.text(x, y, f"V{i}", ha="center", va="center",
            fontsize=10, fontweight="bold", zorder=4)
    lx, ly = x * 1.32, y * 1.32
    ax.text(lx, ly, f"V{i}", ha="center", va="center", fontsize=8,
            color="#333333")

# 凡例
from matplotlib.lines import Line2D
legend_elems = [
    Line2D([0], [0], color="#d1495b", lw=2, linestyle="-",
           label="positive link (+)"),
    Line2D([0], [0], color="#3a6ea5", lw=2, linestyle="--",
           label="negative link (-)"),
]
ax.legend(handles=legend_elems, loc="upper right", fontsize=9)

ax.set_title(f"Causal Loop Diagram (Quantum Computing)\n"
             f"Reinforcing R={r_count}, Balancing B={b_count}",
             fontsize=12)
ax.set_xlim(-1.6, 1.6)
ax.set_ylim(-1.6, 1.6)
ax.set_aspect("equal")
ax.axis("off")

# 変数IDの対応をテキストで添える
note = "\n".join(f"V{i}: {VARIABLES[i]}" for i in range(n))
ax.text(-1.55, -1.55, note, fontsize=7, va="bottom", ha="left")

plt.tight_layout()
plt.show()

## ループが生む動的な振る舞い (behavior-over-time)

ヒートマップは、ショックによる新しい影響が「いつ、どの変数へ到着したか」を示した。ここでは同じ2025〜2040年のショック伝播結果を使い、到着した影響がその後どれだけ残り、いつピークを迎え、いつ減衰するかを見る。V0〜V5（利用・データ・人材・価格系統）と V6〜V9（FTQC・暗号リスク・PQC系統）を分けて表示する。

変数間では影響量の桁が異なるため、各線はその変数自身の最大絶対影響で割り、-1〜+1の相対応答として表示する。したがって線の高さは変数同士の絶対的な影響量比較ではなく、**初動・ピーク・持続・反転の時期比較**に用いる。実際のピーク値と累積影響は凡例および出力表に残す。

In [ ]:
# 動的分析もヒートマップと同じ年次ショックモデルを使う。
# V0〜V5には利用拡大シナリオ、V6〜V9にはFTQC進展シナリオを対応させる。
FOCUS = 0
behavior = np.zeros((len(YEARS), len(VARIABLES)))
behavior[:, 0:6] = scenario_results[0]["impact"][:, 0:6]
behavior[:, 6:10] = scenario_results[1]["impact"][:, 6:10]

In [ ]:
# 同一のbehaviorを、因果的に独立した2系統へ分けて表示する。
groups = [
    (range(0, 6), "V0–V5: usage, data, workforce and price"),
    (range(6, 10), "V6–V9: FTQC, cryptographic risk and PQC"),
]
fig, axes = plt.subplots(1, 2, figsize=(18, 6.5), sharex=True, sharey=True)
cmap = plt.get_cmap("tab10")
summary = []

for ax, (actor_ids, title) in zip(axes, groups):
    for i in actor_ids:
        series = behavior[:, i]
        peak_idx = int(np.argmax(np.abs(series)))
        peak_value = series[peak_idx]
        scale = max(abs(peak_value), 1e-12)
        relative = series / scale

        active = np.where(np.abs(series) >= 0.05)[0]
        onset_year = int(YEARS[active[0]]) if len(active) else None
        convergence_year = None
        for t in range(peak_idx + 1, len(YEARS)):
            if np.all(np.abs(series[t:]) < 0.10):
                convergence_year = int(YEARS[t])
                break

        ax.plot(YEARS, relative, lw=2.0, color=cmap(i % 10),
                label=(f"V{i}: {VARIABLES[i]} "
                       f"(peak {peak_value:+.2f}, {YEARS[peak_idx]})"))
        ax.scatter(YEARS[peak_idx], relative[peak_idx], s=35,
                   color=cmap(i % 10), edgecolor="white", zorder=3)
        summary.append((i, onset_year, int(YEARS[peak_idx]), peak_value,
                        convergence_year, float(np.sum(series))))

    ax.axhline(0, color="#666666", lw=1)
    ax.axvline(SHOCK_SCENARIOS[0]["year"], color="#777777",
               ls="--", lw=1, label="shock year")
    ax.set_title(title)
    ax.set_xlabel("year")
    ax.set_ylim(-1.12, 1.12)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5))

axes[0].set_ylabel("relative response (own absolute peak = 1)")
fig.suptitle("Behavior over time after a one-time shock (2025–2040)", fontsize=13)
plt.tight_layout()
plt.show()

print("応答タイミングの要約")
print("ID  初動年  ピーク年  ピーク値  収束年  累積影響")
print("-" * 62)
for i, onset, peak_year, peak_value, convergence, cumulative in summary:
    onset_text = str(onset) if onset is not None else "—"
    convergence_text = str(convergence) if convergence is not None else "2040以降"
    print(f"V{i:<2} {onset_text:>6}  {peak_year:>7}  {peak_value:>+8.2f}  "
          f"{convergence_text:>8}  {cumulative:>+8.2f}")

print("\n[読み方]")
print("  線の高さは各変数内で正規化されており、変数間の絶対量比較には使わない。")
print("  ピークの左右位置で応答の早さ、線の幅で影響の持続性を比較する。")
print("  実際の影響量は凡例のpeak値と要約表で確認する。")

## ループ・ドミナンスの交代 (loop dominance shift)

V0へ戻る影響を、強化ループR1の `V2→V0` と均衡ループB1の `V5→V0` に分解する。ショック伝播時にリンク別の到着量を記録し、それぞれを共通の残存率で翌年へ持ち越すため、各年にV0へ残っているR/B寄与を直接比較できる。B寄与がR寄与を初めて上回る年をドミナンス交代年とするが、係数によっては交代しない。その場合も「現在の制約ループは成長を逆転させるほど強くない」という分析結果になる。

In [ ]:
def edge_loop_types(link_keys, cycles):
    """各リンクが強化(R)・均衡(B)のどちらの閉路に属するかを返す。"""
    types = {key: set() for key in link_keys}
    for nodes, sign_product in cycles:
        kind = "R" if sign_product > 0 else "B"
        for k in range(len(nodes)):
            edge = (nodes[k], nodes[(k + 1) % len(nodes)])
            if edge in types:
                types[edge].add(kind)
    return types


# 利用拡大シナリオで、V0へ戻るリンク別の残存寄与を比較する。
dominance_result = scenario_results[0]
link_keys = dominance_result["link_keys"]
loop_types = edge_loop_types(link_keys, cycles)
r_c = np.zeros(len(YEARS))
b_c = np.zeros(len(YEARS))

for e, (src, dst) in enumerate(link_keys):
    if dst != FOCUS:
        continue
    contribution = dominance_result["edge_impact"][:, e]
    if "R" in loop_types[(src, dst)]:
        r_c += np.maximum(contribution, 0.0)
    if "B" in loop_types[(src, dst)]:
        b_c += np.maximum(-contribution, 0.0)
    print(f"V{src}->V{dst}: loop={','.join(sorted(loop_types[(src, dst)])) or 'none'}, "
          f"peak contribution={contribution[np.argmax(np.abs(contribution))]:+.3f}")

# B寄与がR寄与を初めて上回る実年を探す。
handover_idx = None
for t in range(1, len(YEARS)):
    if r_c[t - 1] >= b_c[t - 1] and r_c[t] < b_c[t] and b_c[t] > 1e-9:
        handover_idx = t
        break
handover_year = int(YEARS[handover_idx]) if handover_idx is not None else None

print(f"強化ループRの最大残存寄与: {r_c.max():.4f}")
print(f"均衡ループBの最大残存寄与: {b_c.max():.4f}")
print(f"ドミナンス交代年: {handover_year}" if handover_year is not None
      else "ドミナンス交代なし: 現在の係数ではR寄与がB寄与を下回らない")

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(11, 7), sharex=True,
    gridspec_kw={"height_ratios": [2, 1]}
)

# 上段: ヒートマップと同じ年次モデルによるV0の通常ケースとの差
focus_series = dominance_result["impact"][:, FOCUS]
ax1.plot(YEARS, focus_series, color="#222222", lw=2.6,
         label=f"V{FOCUS}: {VARIABLES[FOCUS]}")
ax1.axhline(0, color="#777777", lw=1)
ax1.set_ylabel("difference from baseline (points)")
ax1.set_title(f"Annual loop dominance on V{FOCUS} ({VARIABLES[FOCUS]})")
ax1.grid(alpha=0.3)

# 下段: V0へ戻ってきたR/Bリンクの残存寄与
ax2.plot(YEARS, r_c, color="#d1495b", lw=2.0,
         label="reinforcing contribution (R1)")
ax2.plot(YEARS, b_c, color="#3a6ea5", lw=2.0,
         label="balancing contribution (B1, absolute)")
ax2.fill_between(YEARS, r_c, b_c, where=(r_c >= b_c),
                 color="#d1495b", alpha=0.15, label="R dominates")
ax2.fill_between(YEARS, r_c, b_c, where=(b_c > r_c),
                 color="#3a6ea5", alpha=0.15, label="B dominates")
ax2.set_xlabel("year")
ax2.set_ylabel("remaining contribution")
ax2.grid(alpha=0.3)

if handover_year is not None:
    for ax in (ax1, ax2):
        ax.axvline(handover_year, color="#5a5a5a", ls="--", lw=1.5)
    ax1.annotate(
        "dominance handover (R -> B)",
        xy=(handover_year, focus_series[handover_idx]),
        xytext=(handover_year + 1, focus_series[handover_idx]),
        arrowprops=dict(arrowstyle="->", color="#5a5a5a")
    )

ax1.legend(fontsize=8)
ax2.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("[解釈]")
if handover_year is None:
    print("  現在の伝播係数では強化側が優勢で、価格制約は成長を弱めるが逆転させない。")
    print("  B1の係数を強める、または遅延を短くすると交代が起こり得る。")
else:
    print(f"  {handover_year}年にB1の残存寄与がR1を上回る。")
print("  この判定はヒートマップと同じ年次・係数・遅延・残存率に基づく。")

## レバレッジポイントの特定 (sensitivity analysis)

ドネラ・メドウズは、システムには「小さな介入が大きな構造変化を生む点（レバレッジポイント）」が存在すると論じた。ここでは共通 `LINKS` の各 `effect` を±15%変え、同じ2025〜2040年・同じショック・同じ遅延で再計算する。2040年には影響がほぼ消えている可能性があるため、最終年の一点ではなく、対応するサブシステムに期間中残った**累積絶対影響**の変化量を感度とする。感度が大きいリンクほど、小さな係数変化がシステム全体の応答を大きく変える。

In [ ]:
def cumulative_system_impact(result, actor_ids):
    """対象サブシステムの期間内累積絶対影響を返す。"""
    return float(np.sum(np.abs(result["impact"][:, list(actor_ids)])))


def link_sensitivities(links, scenarios, delta=0.15):
    """各effectを±delta変え、同じ年次モデルの累積影響差を測る。"""
    keys = list(links.keys())
    sensitivities = []
    low_metrics = []
    high_metrics = []

    for key in keys:
        src, dst = key
        if src <= 5 and dst <= 5:
            scenario = scenarios[0]
            actor_ids = range(0, 6)
        else:
            scenario = scenarios[1]
            actor_ids = range(6, 10)

        metrics = []
        for factor in (1.0 - delta, 1.0 + delta):
            modified = {k: dict(v) for k, v in links.items()}
            modified[key]["effect"] *= factor
            result = simulate_shock_propagation(
                YEARS, len(VARIABLES), modified,
                scenario["year"], scenario["actor"], scenario["size"],
                PERSISTENCE
            )
            metrics.append(cumulative_system_impact(result, actor_ids))

        low_metrics.append(metrics[0])
        high_metrics.append(metrics[1])
        sensitivities.append(abs(metrics[1] - metrics[0]))

    return keys, np.array(sensitivities), np.array(low_metrics), np.array(high_metrics)


sensitivity_keys, sens, metric_low, metric_high = link_sensitivities(
    LINKS, SHOCK_SCENARIOS
)
ranking = np.argsort(-sens)
print("レバレッジ・ランキング")
print("指標: effectを±15%変えたときの、対応サブシステムの累積絶対影響差")
print("-" * 72)
for rank, k in enumerate(ranking, start=1):
    src, dst = sensitivity_keys[k]
    p = LINKS[(src, dst)]
    subsystem = "V0–V5" if src <= 5 and dst <= 5 else "V6–V9"
    print(f"  {rank:>2}. V{src}->V{dst} ({'+' if p['sign'] > 0 else '-'}) "
          f"[{subsystem}] {VARIABLES[src]} -> {VARIABLES[dst]}")
    print(f"      感度={sens[k]:.4f}  low={metric_low[k]:.3f}  high={metric_high[k]:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"V{sensitivity_keys[k][0]}->V{sensitivity_keys[k][1]}"
          for k in ranking]
vals = sens[ranking]
colors = [
    "#e07a3f" if sensitivity_keys[k][0] <= 5 else "#5b8db8"
    for k in ranking
]
ax.barh(range(len(vals)), vals, color=colors)
ax.set_yticks(range(len(vals)))
ax.set_yticklabels(labels, fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("sensitivity of cumulative absolute impact")
ax.set_title("Leverage points under the common annual shock model")
ax.grid(axis="x", alpha=0.3)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#e07a3f", label="V0–V5 subsystem"),
    Patch(facecolor="#5b8db8", label="V6–V9 subsystem"),
], fontsize=8)
plt.tight_layout()
plt.show()

top_k = ranking[0]
ts, td = sensitivity_keys[top_k]
print("[解釈]")
print(f"  最も感度の高いリンクは V{ts}->V{td}"
      f"（{VARIABLES[ts]} -> {VARIABLES[td]}）。")
print("  これは2040年の一点ではなく、2025〜2040年に残った影響全体への感度である。")
print("  全リンクを同じLINKS・年次・遅延・ショック条件で比較している。")

## システム原型の検出 (system archetypes)

システム原型は、異なる文脈で繰り返し現れるループ構造のパターンである。代表が「成長の限界（limits to growth）」——強化ループによる成長が、ある制約ループによって必ず頭打ちになる構造。ここでは検出した全ループから、強化ループと均衡ループが少なくとも1つの変数を共有する組を探し、limits-to-growth 原型として列挙する。

In [ ]:
def detect_limits_to_growth(cycles):
    """強化ループと均衡ループが変数を共有する組を limits-to-growth として検出。"""
    r_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg > 0]
    b_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg <= 0]
    found = []
    for ri, rset in r_loops:
        for bi, bset in b_loops:
            shared = rset & bset
            if shared:
                found.append((ri, bi, sorted(shared)))
    return found


def detect_shifting_the_burden(cycles):
    """参考: 2 本以上の均衡ループが変数を共有する『問題のすり替え』の簡易判定。"""
    b_loops = [(i, set(nd)) for i, (nd, sg) in enumerate(cycles) if sg <= 0]
    found = []
    for a in range(len(b_loops)):
        for b in range(a + 1, len(b_loops)):
            shared = b_loops[a][1] & b_loops[b][1]
            if shared:
                found.append((b_loops[a][0], b_loops[b][0], sorted(shared)))
    return found


ltg = detect_limits_to_growth(cycles)
print(f"検出された『成長の限界 (limits to growth)』原型: {len(ltg)} 組")
print("=" * 64)
for ri, bi, shared in ltg:
    r_nodes, _ = cycles[ri]
    b_nodes, _ = cycles[bi]
    print(f"  強化ループ: {' -> '.join('V%d' % v for v in r_nodes)} -> (先頭へ)")
    print(f"  均衡ループ: {' -> '.join('V%d' % v for v in b_nodes)} -> (先頭へ)")
    print(f"  共有変数  : {', '.join(VARIABLES[v] for v in shared)}")
    print("-" * 64)

stb = detect_shifting_the_burden(cycles)
print(f"参考 — 均衡ループ同士が変数を共有する組（問題のすり替えの芽）: {len(stb)} 組")

In [ ]:
print("[解釈] システム原型としての読み")
print()
print("量子題材では『QPU時間↑ → 量子人材求人数↑ → 年間報酬中央値↑ → クラウド実効単価↑ → QPU時間↓』というコスト制約ループが、強化ループ R1（QPU時間→蓄積実行データ量→基準問題成功率→QPU時間）の成長を抑える『成長の限界』にあたる。蓄積実行データ量というストックが強化側の駆動源、人材・単価の連鎖が均衡側の制約となる。")
print()
if ltg:
    ri, bi, shared = ltg[0]
    sv = VARIABLES[shared[0]]
    print(f"  検出された原型の核となる共有変数は『{sv}』。")
    print("  強化ループが成長を駆動する一方、同じ変数を通る均衡ループが")
    print("  制約として働き、成長は必ず頭打ちになる。")
    print("  対策の定石は『成長を速める』ことではなく『制約ループを緩める』こと —")
    print("  すなわちレバレッジポイント分析(C)で上位に来たリンクへの介入である。")
else:
    print("  このモデルでは limits-to-growth 原型は検出されなかった。")

## 未来デザイン論文での使われ方と結論への影響

システムズシンキングは、未来デザイン論文において「何が起きるか」を当てる予測装置としてではなく、「どこを動かせば望ましい方向へ系を導けるか」を論じる介入設計の装置として用いられる。論文は典型的に、対象技術をめぐる主要変数を因果ループ図に編成し、強化ループと均衡ループ、そしてリンクに伴う遅延を提示する。そのうえで、なぜ単純な線形外挿が誤るのか——フィードバックが効果を増幅または減衰させ、遅延が原因と結果を時間的に切り離すから——を構造的に説明し、結論を「予測値の提示」から「レバレッジポイントの特定」へと組み替える。読者に手渡されるのは将来の数値ではなく、最小の介入で最大の構造変化を生む地点の地図である。

この手法がもたらす結論の型は、おのずと「どこを動かすか」という規範的・処方的な形をとる。境界設定の経路で見れば、結論はモデルに取り込んだ変数とループの内部でのみ成立し、図に描かれなかった要素は最初から論証の射程外に置かれる。時間観の経路では、未来は内生的な構造から生成されるものと捉えられ、過去の延長でも単なる選択対象でもなく「現在のループ構造がほどけていく先」として描かれる。価値の所在は、何を変数とし何をレバレッジと呼ぶかという構造の切り取り方そのものに埋め込まれる。

同時に、この手法は固有のバイアスを結論に持ち込む。内生的なフィードバック構造を重視するため、モデル境界の外から来る外生ショック——突発的な規制転換、地政学的断絶、無関係な技術の波及——は構造的に過小評価されやすい。また、ループの存在を強調する論の運びは、意図せざる結果や政策抵抗（介入が均衡ループを刺激して打ち消される現象）を前景化させる方向に結論を傾け、「素朴な介入はうまくいかない」という慎重論へ自然に着地しがちである。論文がこの手法を採るとき、結論の説得力はモデル境界の妥当性と外生要因の扱いをどれだけ誠実に明示するかにかかっている。

## 発展課題

**課題A**: `LINKS` の `effect`・`delay`、ショックの年・大きさを変え、初動年・ピーク年・収束年がどう変わるか観察せよ。特にR1とB1の係数を変え、V0におけるドミナンス交代が発生する条件を調べよ。

**課題B**: 自分で新しい変数とリンクを `LINKS` に追加し（題材に応じて成長の限界を緩める制約緩和ループや、遅延を伴う独自ループを設計する）、レバレッジ・ランキングと原型検出がどう変わるかを観察せよ。`EDGES` は `LINKS` から自動生成される。追加したリンクが上位レバレッジに食い込むか、新たな limits-to-growth 原型が生まれるかを論じること。